# Lab 21 — constrained threshold operating point

Mục tiêu: tìm threshold thực tế hơn cho hai ứng viên sau Lab 19/20.

- `age_bin + LightGBM`: ứng viên ưu tiên recall.
- `baseline + Logistic Regression`: ứng viên ưu tiên risk score/calibration.
- Thử recall tối thiểu 80%, 85%, 90%.
- Trong các threshold đạt recall mục tiêu trên inner training hospitals, chọn specificity cao nhất.
- Outer test hospital là dữ liệu thật bị khóa; không dùng để chọn threshold.

In [ ]:
!pip -q install lightgbm seaborn

import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
FIXED_THRESHOLD = 0.50
THRESHOLD_GRID = np.arange(0.05, 0.91, 0.01)
TARGET_RECALLS = [0.80, 0.85, 0.90]
OUTPUT_DIR = Path('/content/uci_multicenter_constrained_threshold_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
AGE_BIN_FEATURE = 'age_bin'
AGE_BINS = [0, 39, 49, 59, 69, 120]
AGE_LABELS = [0, 1, 2, 3, 4]
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
COLUMNS = FEATURES + ['num']
EXPERIMENTS = {
    'candidate_age_bin_LightGBM': {'feature_config': 'F2_P1_age_bin', 'model': 'LightGBM'},
    'candidate_baseline_Logistic': {'feature_config': 'F0_P1_baseline', 'model': 'Logistic Regression'},
}

def read_uci(site, filename):
    frame = pd.read_csv(f'{BASE_URL}/{filename}', names=COLUMNS, na_values=['?'],
                        skipinitialspace=True).apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
SITES = list(FILES.keys())
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))

In [ ]:
FEATURE_CONFIGS = {
    'F0_P1_baseline': {'age_bin': False},
    'F2_P1_age_bin': {'age_bin': True},
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def make_features(frame, feature_config):
    out = apply_p1(frame)
    if FEATURE_CONFIGS[feature_config]['age_bin']:
        out['age_bin'] = pd.cut(out['age'], bins=AGE_BINS, labels=AGE_LABELS,
                                include_lowest=True).astype('object')
    return out

def feature_groups(feature_config):
    numeric = NUMERICAL_FEATURES.copy()
    categorical = CATEGORICAL_FEATURES.copy()
    if FEATURE_CONFIGS[feature_config]['age_bin']:
        categorical += [AGE_BIN_FEATURE]
    return numeric, categorical

def build_pipeline(feature_config, model_name):
    numeric_features, categorical_features = feature_groups(feature_config)
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer([('numeric', numeric, numeric_features),
                                     ('categorical', categorical, categorical_features)])
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    else:
        estimator = LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            random_state=RANDOM_STATE, verbosity=-1)
    return Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])

display(pd.DataFrame({name: pd.Series(feature_groups(spec['feature_config'])[0] +
                                   feature_groups(spec['feature_config'])[1])
                      for name, spec in EXPERIMENTS.items()}).T)

## 1. Inner LOCO và constrained threshold

Threshold được chọn bằng OOF probabilities của ba training hospitals. Trong các threshold đạt recall mục tiêu, chọn specificity cao nhất; nếu không đạt được mục tiêu, ghi rõ fallback thay vì dùng outer test để điều chỉnh.

In [ ]:
def safe_auc(y_true, probability):
    return roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan

def expected_calibration_error(y_true, probability, n_bins=10):
    y_true = np.asarray(y_true); probability = np.asarray(probability)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probability >= left) & (probability <= right if right == 1 else probability < right)
        if mask.any():
            ece += mask.mean() * abs(probability[mask].mean() - y_true[mask].mean())
    return float(ece)

def fit_platt_calibrator(probability, y_true):
    clipped = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)
    calibrator = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
    calibrator.fit(logits, y_true)
    return calibrator

def apply_platt_calibrator(calibrator, probability):
    clipped = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)
    return calibrator.predict_proba(logits)[:, 1]

def inner_oof(train_frame, feature_config, model_name):
    groups = train_frame['site'].to_numpy(); y = train_frame[TARGET].to_numpy()
    probabilities = np.full(len(train_frame), np.nan, dtype=float)
    for inner_train_idx, inner_valid_idx in GroupKFold(n_splits=3).split(train_frame, y, groups):
        inner_train = train_frame.iloc[inner_train_idx]
        inner_valid = train_frame.iloc[inner_valid_idx]
        train_ready = make_features(inner_train, feature_config)
        valid_ready = make_features(inner_valid, feature_config)
        numeric_features, categorical_features = feature_groups(feature_config)
        columns = numeric_features + categorical_features
        model = build_pipeline(feature_config, model_name)
        model.fit(train_ready[columns], inner_train[TARGET])
        probabilities[inner_valid_idx] = model.predict_proba(valid_ready[columns])[:, 1]
    assert np.isfinite(probabilities).all(), 'Incomplete inner OOF probabilities'
    return y, probabilities

def constrained_threshold(y_true, probability, target_recall):
    feasible = []
    all_rows = []
    for threshold in THRESHOLD_GRID:
        prediction = (probability >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        precision = precision_score(y_true, prediction, zero_division=0)
        row = {'threshold': float(threshold), 'recall': recall,
               'specificity': specificity, 'precision': precision,
               'feasible': bool(recall >= target_recall)}
        all_rows.append(row)
        if row['feasible']: feasible.append(row)
    if feasible:
        best = max(feasible, key=lambda r: (r['specificity'], r['precision'], r['threshold']))
        best['fallback'] = False
    else:
        best = max(all_rows, key=lambda r: (r['recall'], r['specificity'], r['threshold']))
        best['fallback'] = True
    best['target_recall'] = target_recall
    return best

def score_predictions(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'npv': tn / (tn + fn) if (tn + fn) else np.nan,
        'alert_rate': float(prediction.mean()),
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': safe_auc(y_true, probability),
        'brier': brier_score_loss(y_true, probability),
        'ece': expected_calibration_error(y_true, probability),
        'false_negatives': int(fn), 'false_positives': int(fp),
        'threshold': float(threshold)}


In [ ]:
results = []
threshold_audits = []
for test_site in SITES:
    train_frame = data[data['site'] != test_site].reset_index(drop=True)
    test_frame = data[data['site'] == test_site].reset_index(drop=True)
    for experiment, spec in EXPERIMENTS.items():
        feature_config, model_name = spec['feature_config'], spec['model']
        numeric_features, categorical_features = feature_groups(feature_config)
        columns = numeric_features + categorical_features
        y_oof, raw_oof = inner_oof(train_frame, feature_config, model_name)
        calibrator = fit_platt_calibrator(raw_oof, y_oof)
        calibrated_oof = apply_platt_calibrator(calibrator, raw_oof)
        final_train = make_features(train_frame, feature_config)
        final_test = make_features(test_frame, feature_config)
        final_model = build_pipeline(feature_config, model_name)
        started = time.perf_counter()
        final_model.fit(final_train[columns], train_frame[TARGET])
        fit_seconds = time.perf_counter() - started
        raw_test = final_model.predict_proba(final_test[columns])[:, 1]
        calibrated_test = apply_platt_calibrator(calibrator, raw_test)
        probability_sets = [('raw', raw_oof, raw_test), ('sigmoid_calibrated', calibrated_oof, calibrated_test)]
        for probability_type, oof_probability, test_probability in probability_sets:
            for target_recall in TARGET_RECALLS:
                selected = constrained_threshold(y_oof, oof_probability, target_recall)
                audit = {'test_site': test_site, 'experiment': experiment, 'model': model_name,
                    'probability_type': probability_type, **selected,
                    'inner_oof_brier': brier_score_loss(y_oof, oof_probability),
                    'inner_oof_ece': expected_calibration_error(y_oof, oof_probability)}
                threshold_audits.append(audit)
                metrics = score_predictions(test_frame[TARGET].to_numpy(), test_probability, selected['threshold'])
                results.append({'test_site': test_site, 'experiment': experiment, 'model': model_name,
                    'probability_type': probability_type, 'policy': 'constrained_recall',
                    'target_recall': target_recall, 'test_rows': len(test_frame),
                    'fit_seconds': fit_seconds, **metrics})
            fixed_metrics = score_predictions(test_frame[TARGET].to_numpy(), test_probability, FIXED_THRESHOLD)
            results.append({'test_site': test_site, 'experiment': experiment, 'model': model_name,
                'probability_type': probability_type, 'policy': 'fixed_0.50',
                'target_recall': np.nan, 'test_rows': len(test_frame),
                'fit_seconds': fit_seconds, **fixed_metrics})
    print('Completed outer test site:', test_site)

results_df = pd.DataFrame(results)
threshold_df = pd.DataFrame(threshold_audits)
display(results_df.head())
display(threshold_df.round(4))

## 2. Tổng hợp operating points

Ưu tiên đọc `recall_mean`, `specificity_mean`, `false_negatives_total`, `alert_rate` và `fallback`. Không chọn threshold theo một outer hospital đơn lẻ.

In [ ]:
summary = results_df.groupby(['experiment', 'model', 'probability_type', 'policy', 'target_recall'], dropna=False).agg(
    folds=('test_site', 'nunique'), roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'), roc_auc_worst=('roc_auc', 'min'),
    pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'), recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'), precision_mean=('precision', 'mean'),
    npv_mean=('npv', 'mean'), alert_rate_mean=('alert_rate', 'mean'),
    f1_mean=('f1', 'mean'), brier_mean=('brier', 'mean'), ece_mean=('ece', 'mean'),
    false_negatives_mean_per_fold=('false_negatives', 'mean'),
    false_negatives_total=('false_negatives', 'sum'),
    false_positives_total=('false_positives', 'sum'), threshold_mean=('threshold', 'mean'),
    fit_seconds_mean=('fit_seconds', 'mean')).reset_index()
fallback_summary = threshold_df.groupby(['experiment', 'model', 'probability_type', 'target_recall']).agg(
    fallback_count=('fallback', 'sum')).reset_index()
summary = summary.merge(fallback_summary, on=['experiment', 'model', 'probability_type', 'target_recall'], how='left')
summary['fallback_count'] = summary['fallback_count'].fillna(0).astype(int)

display(summary.sort_values(['experiment', 'probability_type', 'policy', 'target_recall']).round(6))

constrained = summary[summary['policy'] == 'constrained_recall'].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.lineplot(data=constrained, x='target_recall', y='specificity_mean',
             hue='experiment', style='probability_type', markers=True, dashes=False, ax=axes[0])
axes[0].set_title('Specificity at constrained recall targets'); axes[0].set_ylim(0, 1)
sns.lineplot(data=constrained, x='target_recall', y='false_negatives_mean_per_fold',
             hue='experiment', style='probability_type', markers=True, dashes=False, ax=axes[1])
axes[1].set_title('False negatives per outer fold'); axes[1].set_ylim(bottom=0)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'constrained_threshold_operating_points.png', dpi=180, bbox_inches='tight'); plt.show()

results_path = OUTPUT_DIR / 'constrained_threshold_loco_results.csv'
threshold_path = OUTPUT_DIR / 'constrained_threshold_inner_oof_audit.csv'
summary_path = OUTPUT_DIR / 'constrained_threshold_summary.csv'
results_df.to_csv(results_path, index=False)
threshold_df.to_csv(threshold_path, index=False)
summary.to_csv(summary_path, index=False)
run_config = {'dataset_rows': 920, 'validation': 'outer LOCO + inner GroupKFold by hospital',
    'preprocessing': 'P1_sentinel_aware', 'fixed_threshold': FIXED_THRESHOLD,
    'threshold_grid': [0.05, 0.90, 0.01], 'target_recalls': TARGET_RECALLS,
    'selection_rule': 'maximum specificity subject to recall >= target; fallback records infeasibility',
    'calibration': 'Platt sigmoid fitted only on inner OOF probabilities',
    'age_bins': AGE_BINS, 'experiments': EXPERIMENTS,
    'outer_test_policy': 'real held-out hospital only'}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_constrained_threshold_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)